#  顧客留存分析｜舊客活化與再購行為分析

商業問題：  
顧客回購率與購買週期分析  
分析方法：  
- 計算顧客每次訂單間隔天數
- 分析整體購買間隔分佈

In [1]:
import pandas as pd
sales=pd.read_csv("sales.csv")
sales["Order_Date"]=pd.to_datetime(sales["Order_Date"],dayfirst=True)
sales=sales.sort_values(by=["Customer_ID","Order_Date"])
sales["Days_Since_Last_Order"]=sales.groupby("Customer_ID")["Order_Date"].diff().dt.days
desc=sales["Days_Since_Last_Order"].dropna().describe()
print(desc)

count    210086.000000
mean         98.730301
std          94.345220
min           0.000000
25%          29.000000
50%          70.000000
75%         139.000000
max         754.000000
Name: Days_Since_Last_Order, dtype: float64


分析結果：  
回購間隔分析顯示，中位數為 70 天，Q1 為 29 天，代表一半的回購間隔不超過 70 天，25% 不超過 29 天。平均間隔為 98.7 天，高於中位數，顯示回購間隔分布呈右偏，部分顧客存在較長的回購間隔；最大值為 754 天，反映少數顧客的購買週期較長，可進一步搭配最近購買時間與 RFM 分析辨識潛在流失或喚回對象。

商業問題：  
顧客留存與流失分析  
分析方法：  
- 依註冊月份建立 Cohort
- 計算各 Cohort 月留存率
- 建立留存矩陣

In [2]:
import pandas as pd
sales=pd.read_csv("sales.csv")
customers=pd.read_csv("customers.csv")
customers["Registration_Date"]=pd.to_datetime(customers["Registration_Date"])
sales["Order_Date"]=pd.to_datetime(sales["Order_Date"],dayfirst=True)
df=pd.merge(sales,customers[["Customer_ID","Registration_Date"]],on="Customer_ID",how="inner")
df["CohortMonth"]=df["Registration_Date"].dt.to_period("M")
df['OrderMonth']=df["Order_Date"].dt.to_period("M")
df["CohortIndex"]=((df["OrderMonth"].dt.year-df["CohortMonth"].dt.year)*12+(df["OrderMonth"].dt.month-df["CohortMonth"].dt.month))
reg_totals=customers.groupby(customers["Registration_Date"].dt.to_period("M"))["Customer_ID"].nunique()
cohort_counts=df.groupby(["CohortMonth","CohortIndex"])["Customer_ID"].nunique().unstack()
retention_matrix=cohort_counts.divide(reg_totals,axis=0)*100
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
print(retention_matrix.round(2))

CohortIndex     1      2      3      4      5      6      7      8      9      10     11     12     13     14     15     16     17     18     19     20     21     22     23     24     25     26     27     28     29     30     31     32     33     34     35     36
CohortMonth                                                                                                                                                                                                                                                            
2023-06        NaN    NaN    NaN    NaN    NaN    NaN    NaN    NaN    NaN    NaN    NaN  21.78  22.43  23.15  21.60  22.91  21.42  22.67  21.57  21.13  23.03  22.17  22.79  21.39  23.15  22.91  21.34  21.81  23.26  23.26  21.57  19.64  23.35  22.37  21.90  22.67
2023-07        NaN    NaN    NaN    NaN    NaN    NaN    NaN    NaN    NaN    NaN  22.26  22.41  22.05  21.73  23.03  20.93  23.62  22.85  20.72  22.50  21.94  22.79  20.93  21.67  21.61  22.74  24.30  21.61 

分析結果：  
留存率矩陣顯示，從資料可觀察的最早期間開始，各群組的留存率大致落在約 20～24%，一路至第 36 個月未見明顯的持續下降或上升趨勢，僅有小幅波動。值得注意的是，「註冊當月」的留存率全數缺值。經檢視原始資料，顧客自註冊至首次產生訂單之間確實存在空窗期，因此註冊當月沒有訂單紀錄屬於資料本身的特性。

商業問題：  
商品評價與再購行為分析  
分析方法：  
- 計算顧客平均評價與訂單數
- 依評價區間分組
- 比較各組回購率

In [3]:
import pandas as pd
sales=pd.read_csv("sales.csv")
avg_rating=sales.groupby("Customer_ID")["Rating"].mean().round(2)
order_count=sales.groupby("Customer_ID")["Order_ID"].nunique()
cust=pd.DataFrame({"avg_rating":avg_rating,"is_repurchase":order_count>1})
def get_group(r):
    if r>=4:
        return ">=4分"
    elif r>2:
        return ">2分"
    else:
        return "其他"
cust["group"]=cust["avg_rating"].apply(get_group)
result=cust[cust["group"].isin([">=4分",">2分","其他"])].groupby("group")[
    "is_repurchase"].mean()
print(result)

group
>2分     0.993403
>=4分    0.994105
其他      0.865936
Name: is_repurchase, dtype: float64


分析結果：  
三組回購率分別為「 ≥4 分」 99.41% 、「 >2 分」 99.34% ，以及「其他（評分 ≤2 分或無評分）」 86.59% 。低評價或未評分客群的回購率明顯低於其他評價客群，顯示評價可能與顧客再購行為存在關聯，值得進一步關注。

商業問題：  
高價值顧客流失預警與挽回  
分析方法：  
- 合併 RFM 分數與顧客聯絡資訊
- 計算顧客流失風險與挽回優先分數
- 排序建立 Top 50 挽回名單

In [5]:
import pandas as pd
customers=pd.read_csv("customers.csv")
rfm_result=pd.read_csv("rfm_result.csv")
df_1=pd.merge(rfm_result,customers[["Customer_ID","Customer_Name","Email","Phone"]],on="Customer_ID",how="inner")
df_1["Churn_Risk"]=6-df_1["R_Score"]
df_1["Winback_Score"]=df_1["Churn_Risk"]*(df_1["F_Score"]+df_1["M_Score"])
top50_winback=df_1.sort_values(by=["Winback_Score","Monetary"],ascending=[False, False]).head(50)
result_cols=["Customer_ID","Customer_Name","Churn_Risk","Recency","Monetary","R_Score","F_Score","M_Score","Email","Phone"]
top50_list=top50_winback[result_cols]
top50_list.to_csv("top50_winback_list.csv", index=False, encoding="utf-8-sig")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
print(top50_list)

        Customer_ID    Customer_Name  Churn_Risk  Recency   Monetary  R_Score  F_Score  M_Score                         Email           Phone
19684  CUST00019830    Preeti Mishra           5      272  661090.40        1        5        5    preetimishra23@outlook.com  +91-9839965395
30162  CUST00030374  Shreya Kulkarni           5      259  593804.19        1        5        5   shreyakulkarni739@yahoo.com  +91-8809739822
23661  CUST00023830      Rohan Yadav           5      287  574045.03        1        5        5        rohanyadav@outlook.com  +91-8878086136
25046  CUST00025222    Divya Agarwal           5      264  540537.98        1        5        5     divyaagarwal223@gmail.com  +91-8030004789
16535  CUST00016651      Pooja Verma           5      250  506850.40        1        5        5          poojaverma@yahoo.com  +91-8758532595
11134  CUST00011216     Anjali Mehta           5      406  505734.42        1        5        5      anjalimehta922@gmail.com  +91-7456120887
22353 

分析結果：  
Top 50 清單全數為 R_Score=1、F_Score=5、M_Score=5 的顧客，即過去消費頻率與金額皆高，但距離最近一次購買已較久（ Recency 介於 242～423 天）。依據流失風險與過去消費頻率、金額所建立的 Winback Score ，這批顧客皆具有最高的 Churn Risk=5 ，且過去消費貢獻度高（ Monetary 平均約 40.6 萬，最高達 66.1 萬），因此列為 Top 50 優先挽回對象。